# Agentic RAG: Router–Retriever System (Anthropic + Docling + Local Embeddings)

This notebook implements a multi-agent Retrieval-Augmented Generation (RAG) system using **CrewAI**, running entirely on **Claude** for generation and **local HuggingFace embeddings** for retrieval — no OpenAI key required.


| Stage | Tool |
|-------|--------------|
| Agent LLM | **Claude** (`ANTHROPIC_API_KEY`) |
| PDF reading | **Docling** structured parser |
| Embeddings | **Local** `all-MiniLM-L6-v2` (no key, no quota) |
| Vector store | **ChromaDB** (explicit, inspectable) |
| Web search | Tavily |


## System Architecture

```
User Query
    │
    ▼
┌─────────────────────────────────┐
│ Router Agent  (Claude)          │
│ Classifies query into:          │
│   - pdf_search                  │
│   - web_search                  │
│   - generate                    │
└──────────────┬──────────────────┘
               │ routing decision
               ▼
┌─────────────────────────────────┐
│ Retriever Agent  (Claude)       │
│ Executes retrieval via:         │
│   - Docling+Chroma PDF tool     │
│   - Tavily live web search      │
│   - LLM knowledge (generate)    │
└──────────────┬──────────────────┘
               │ grounded answer
               ▼
          Final Answer
```

### The retrieval pipeline, unpacked (for the PM audience)

The original `PDFSearchTool` is a sealed box that hides five stages. We open the box so each stage is a labeled step you can point at on a slide:

```
Docling  →  Chunk  →  Local embed  →  Chroma  →  BaseTool
(parse)     (split)   (HF MiniLM)     (store +   (so the agent
                                       search)    can call it)
```


## 1. Install Dependencies

Versions are pinned so a live demo doesn't break on an upstream release. Docling pulls in some heavyweight dependencies (PyTorch, layout models) — the install cell takes a minute or two on a fresh Colab runtime.

In [ ]:
# Core: CrewAI + Anthropic routing, Docling parser, local embeddings, Chroma, Tavily
!pip install -q \
    "crewai==0.86.0" \
    "crewai-tools==0.25.8" \
    "docling==2.15.1" \
    "langchain-huggingface==0.1.2" \
    "sentence-transformers==3.3.1" \
    "chromadb==0.5.23" \
    "langchain-community==0.3.14" \
    "tavily-python==0.5.0" \
    "anthropic==0.42.0"

print("\n✅ Dependencies installed. If this is Colab, restart the runtime now:")
print("   Runtime ▸ Restart session, then run from Section 2.")
print("   (A restart clears the stale module cache after big pip installs.)")

> **⚠️ Colab gotcha:** After the install above finishes, do **Runtime ▸ Restart session** before continuing. A `ModuleNotFoundError` for `langchain_huggingface` almost always means a skipped install cell or a stale kernel — restarting fixes it. You do *not* need to re-run the install after restarting.

## 2. Environment Setup & API Keys

This version reads secrets from **Colab Secrets** (the 🔑 panel on the left), not a `.env` file.

Add two secrets and toggle **Notebook access** on for each:

| Secret name | Used for | Required? |
|-------------|----------|-----------|
| `ANTHROPIC_API_KEY` | Claude (both agents) | **Yes** |
| `TAVILY_API_KEY` | Live web search path | Only if you demo `web_search` |

> **Naming gotcha:** the Anthropic SDK expects the variable to be `ANTHROPIC_API_KEY`. `CLAUDE_API_KEY` will silently not be picked up.

In [ ]:
import os

# In Colab, pull secrets from userdata. Locally, fall back to existing env vars.
try:
    from google.colab import userdata

    def _load(name, required=True):
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            os.environ[name] = val
        elif required:
            print(f"⚠️  Secret '{name}' not found in Colab Secrets (🔑 panel).")
        return val

    _load("ANTHROPIC_API_KEY", required=True)
    _load("TAVILY_API_KEY", required=False)
    _env = "Colab Secrets"
except ImportError:
    # Not on Colab — assume keys are already exported in the shell environment.
    _env = "local environment variables"

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY is not set. Add it to Colab Secrets (🔑) and enable "
    "Notebook access, then re-run this cell."
)

# Path to the PDF the retriever will index. Upload it to the Colab file browser,
# or adjust this path. The transformer paper is a good demo doc — dense with
# tables and multi-column layout, which is exactly where Docling shines.
PDF_PATH = "transformer_research_paper-dataset.pdf"

print(f"✅ Environment configured using {_env}.")
print(f"   ANTHROPIC_API_KEY: {'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING'}")
print(f"   TAVILY_API_KEY:    {'set' if os.environ.get('TAVILY_API_KEY') else 'not set (web_search disabled)'}")

## 3. Import Libraries & Set Up Logging

The logging block is unchanged from the original — it writes a full reasoning trace to both a file and stdout, which the project rubric asks for.

In [ ]:
import logging
import json
from datetime import datetime, timezone
from textwrap import dedent

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from pydantic import Field

# ── Logging setup ────────────────────────────────────────────────
LOG_FILE = "agentic_rag_trace.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("AgenticRAG")
logger.info("Logger initialised — Agentic RAG session started.")
print("✅ Imports and logging ready.")

## 4. Configure the Claude LLM for both agents

CrewAI talks to model providers through LiteLLM, so pointing it at Claude is a one-line config change. We wrap it in a `crewai.LLM` object and hand the *same* instance to both agents.

The model string uses the `anthropic/` provider prefix that LiteLLM expects.

In [ ]:
# One Claude instance, shared by both agents.
# Swap the model string for a bigger Claude if you want the Retriever to be
# stronger than the Router (the original notebook made the same note about
# using a more powerful model for retrieval in production).
claude_llm = LLM(
    model="anthropic/claude-sonnet-4-5-20250929",
    temperature=0.2,
)

logger.info("Claude LLM configured: %s", claude_llm.model)
print(f"✅ Claude LLM ready: {claude_llm.model}")

## 5. Build the PDF retrieval pipeline (Docling → chunk → local embed → Chroma)

This section replaces `PDFSearchTool` entirely. Each of the five stages is explicit and inspectable.

**5a. Parse with Docling.** Docling does document *structure* recognition — tables, headings, reading order — not just a flat text dump. For a multi-column research paper that means the extracted Markdown chunks cleanly along real section boundaries instead of splicing columns together.

In [ ]:
from docling.document_converter import DocumentConverter

def parse_pdf_to_markdown(pdf_path: str) -> str:
    """Parse a PDF into structure-aware Markdown using Docling (v2 API)."""
    logger.info("[Docling] Parsing PDF: %s", pdf_path)
    converter = DocumentConverter()
    result = converter.convert(pdf_path)          # v2 API: convert() ...
    markdown = result.document.export_to_markdown()  # ... then export_to_markdown()
    logger.info("[Docling] Parsed %d characters of Markdown.", len(markdown))
    return markdown

# First Docling run downloads its layout models (~tens of MB) and takes ~30-60s.
pdf_markdown = parse_pdf_to_markdown(PDF_PATH)
print(f"✅ Docling parsed the PDF into {len(pdf_markdown):,} characters of Markdown.")
print("\n─── First 600 chars preview ───")
print(pdf_markdown[:600])

**5b. Chunk.** Split the Markdown into overlapping passages. Overlap keeps a sentence that straddles a boundary from being cut in half. We chunk on the Markdown so structure is respected.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(pdf_markdown)
logger.info("[Chunk] Split Markdown into %d chunks.", len(chunks))
print(f"✅ Split into {len(chunks)} chunks (size≈1000, overlap=150).")

**5c. Embed locally + store in Chroma.** The embedding model runs on the Colab CPU — no API key, no quota, no remote endpoint to 429 mid-demo. This is the single change that kills the original notebook's `insufficient_quota` crash.

In [ ]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

# Local embedding model — downloads once (~90 MB), then runs offline on CPU.
embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
logger.info("[Embed] Local embedding model loaded: all-MiniLM-L6-v2")

# In-memory Chroma collection (resets each session — fine for a demo).
chroma_client = chromadb.Client()
COLLECTION_NAME = "pdf_chunks"
# Drop any existing collection so re-running the cell is idempotent.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(COLLECTION_NAME)

# Embed all chunks locally and add them to Chroma.
embeddings = embedder.embed_documents(chunks)
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings,
)
logger.info("[Chroma] Stored %d embedded chunks.", len(chunks))
print(f"✅ Embedded and stored {len(chunks)} chunks in Chroma (local, no API key).")

**5d. Wrap search as a CrewAI tool.** Finally we expose similarity search as a `BaseTool` so the Retriever agent can call it — the same interface the original `PDFSearchTool` presented to the agent, but now backed by our own transparent pipeline.

In [ ]:
class DoclingPDFSearchTool(BaseTool):
    """Semantic search over the Docling-parsed, locally-embedded PDF chunks."""
    name: str = "pdf_search"
    description: str = (
        "Searches the local PDF corpus (a transformer research paper) for "
        "domain-specific, static content. Use this for questions about the "
        "document's subject matter."
    )

    def _run(self, query: str) -> str:
        logger.info("[DoclingPDFSearchTool] Query: %s", query)
        q_emb = embedder.embed_query(query)
        results = collection.query(query_embeddings=[q_emb], n_results=4)
        docs = results.get("documents", [[]])[0]
        if not docs:
            return "No relevant passages found in the PDF."
        logger.info("[DoclingPDFSearchTool] Returned %d passages.", len(docs))
        return "\n\n---\n\n".join(
            f"[Passage {i+1}]\n{d}" for i, d in enumerate(docs)
        )

pdf_search_tool = DoclingPDFSearchTool()
logger.info("DoclingPDFSearchTool initialised.")
print("✅ PDF search tool ready (Docling + local embeddings + Chroma).")

## 6. Web Search Tool (Tavily)

Unchanged from the original — a CrewAI-compatible wrapper around Tavily. This is the one remaining path that needs a paid key (`TAVILY_API_KEY`). If you skipped that secret, the `web_search` route will raise when exercised; the `pdf_search` and `generate` routes still work.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

class TavilyWebSearchTool(BaseTool):
    """CrewAI-compatible wrapper around TavilySearchResults."""
    name: str = "web_search"
    description: str = (
        "Searches the internet for up-to-date information. Use this when the "
        "question requires recent or real-world knowledge not in the local PDF."
    )
    _client: TavilySearchResults = None

    def model_post_init(self, __context):
        self._client = TavilySearchResults(max_results=5)

    def _run(self, query: str) -> str:
        logger.info("[TavilyWebSearchTool] Query: %s", query)
        results = self._client.invoke(query)
        formatted = [
            f"Source: {r.get('url', 'N/A')}\nSnippet: {r.get('content', '')}"
            for r in results
        ]
        logger.info("[TavilyWebSearchTool] Returned %d results.", len(results))
        return "\n\n".join(formatted)

web_search_tool = TavilyWebSearchTool()
logger.info("TavilyWebSearchTool initialised.")
print("✅ Both retrieval tools are ready.")

## 7. Define the Agents

Two agents, both backed by the shared Claude LLM. Note the explicit `llm=claude_llm` on each — that's what routes them to Claude instead of CrewAI's default.

**Router Agent** — classifies the query into `pdf_search`, `web_search`, or `generate`.
**Retriever Agent** — acts on the routing decision, calls the right tool, and composes a grounded answer.

In [ ]:
# ── 7a. Router Agent ──────────────────────────────────────────────
router_agent = Agent(
    role="Query Router",
    goal=dedent("""\
        Analyse the user's question and decide the most appropriate retrieval strategy.
        Output ONLY one of the following labels (no extra text):
          - pdf_search : question is about domain-specific, static content likely in the PDF
          - web_search : question requires current, real-world, or internet-sourced information
          - generate   : question can be answered directly from the LLM's general knowledge
    """),
    backstory=dedent("""\
        You are an expert information architect specialising in query intent classification.
        You know when static document retrieval, live web search, or pure language-model
        generation is the right approach. You are concise and always produce exactly one label.
    """),
    llm=claude_llm,
    verbose=True,
    allow_delegation=False,
)
logger.info("Router Agent created.")

# ── 7b. Retriever Agent ───────────────────────────────────────────
retriever_agent = Agent(
    role="Information Retriever",
    goal=dedent("""\
        Given a routing label and the original user question, retrieve relevant information
        using the correct tool and compose a clear, accurate, source-grounded answer.
          - If the label is 'pdf_search' -> use the pdf_search tool.
          - If the label is 'web_search' -> use the web_search tool.
          - If the label is 'generate'   -> answer directly from your own knowledge.
        Always cite the source of your information in the final answer.
    """),
    backstory=dedent("""\
        You are a meticulous research analyst who extracts precise information from documents
        and the web. You ground answers in retrieved evidence and clearly indicate the source.
    """),
    tools=[pdf_search_tool, web_search_tool],
    llm=claude_llm,
    verbose=True,
    allow_delegation=False,
)
logger.info("Retriever Agent created.")
print("✅ Both agents defined (backed by Claude).")

## 8. Define the Tasks

The routing task feeds its output to the retrieval task as context — the reason the crew runs sequentially rather than in parallel.

In [ ]:
def build_tasks(user_question: str):
    """Construct routing and retrieval tasks for a given user question."""
    routing_task = Task(
        description=dedent(f"""\
            A user has asked the following question:
            "{user_question}"

            Classify this question and respond with EXACTLY ONE of these labels:
              pdf_search  — the answer is likely in the domain-specific PDF document
              web_search  — the answer requires current internet information
              generate    — the answer can be given directly by the LLM

            Do NOT include any explanation, punctuation, or extra words — just the label.
        """),
        expected_output="One of: pdf_search | web_search | generate",
        agent=router_agent,
    )

    retrieval_task = Task(
        description=dedent(f"""\
            The Router Agent has provided a routing decision (see context above).
            The original user question is:
            "{user_question}"

            Follow the routing decision precisely:
              - pdf_search -> call the pdf_search tool with the question as input
              - web_search -> call the web_search tool with the question as input
              - generate   -> answer directly from your LLM knowledge

            Compose a comprehensive, well-structured answer.
            Clearly state which source (PDF, Web, or LLM knowledge) was used.
        """),
        expected_output="A clear, source-grounded answer to the user's question.",
        agent=retriever_agent,
        context=[routing_task],  # passes Router output as context
    )
    return routing_task, retrieval_task

print("✅ Task builder function ready.")

## 9. Assemble the Crew & Run Pipeline

In [ ]:
def run_agentic_rag(user_question: str) -> dict:
    """Execute the full Agentic RAG pipeline for a single user question."""
    logger.info("=" * 60)
    logger.info("New query received: %s", user_question)

    routing_task, retrieval_task = build_tasks(user_question)
    crew = Crew(
        agents=[router_agent, retriever_agent],
        tasks=[routing_task, retrieval_task],
        process=Process.sequential,  # Router first, then Retriever
        verbose=True,
    )
    result = crew.kickoff()

    route_label = (
        routing_task.output.raw.strip().lower()
        if routing_task.output else "unknown"
    )
    final_answer = result.raw if hasattr(result, "raw") else str(result)

    logger.info("Route chosen : %s", route_label)
    preview = final_answer[:300] + ("..." if len(final_answer) > 300 else "")
    logger.info("Final answer : %s", preview)
    logger.info("=" * 60)

    return {
        "question": user_question,
        "route": route_label,
        "answer": final_answer,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

print("✅ run_agentic_rag() function ready.")

## 10. Demo — Run Three Example Queries

| Query | Expected Route |
|-------|----------------|
| What is Agentic RAG and how does a Router Agent work? | `pdf_search` |
| What are the latest breakthroughs in AI announced this week? | `web_search` |
| What is the capital of France? | `generate` |

> If you skipped `TAVILY_API_KEY`, the second query will error on the web path — drop it from `demo_questions` to run a key-free demo.

In [ ]:
demo_questions = [
    "What is Agentic RAG and how does a Router Agent work?",   # -> pdf_search
    "What are the latest breakthroughs in AI announced this week?",  # -> web_search
    "What is the capital of France?",                          # -> generate
]

results = []
for i, question in enumerate(demo_questions, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}: {question}")
    print('='*70)
    output = run_agentic_rag(question)
    results.append(output)
    print(f"\n📍 Route  : {output['route']}")
    ans = output['answer']
    print(f"💬 Answer : {ans[:500]}..." if len(ans) > 500 else f"💬 Answer : {ans}")

## 11. Display Structured Results

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "#": i + 1,
        "Question": r["question"],
        "Route": r["route"],
        "Answer (truncated)": r["answer"][:120] + "...",
        "Timestamp": r["timestamp"],
    }
    for i, r in enumerate(results)
])
pd.set_option("display.max_colwidth", 130)
print("\n📊 Results Summary")
print(df.to_string(index=False))

## 12. Save Results & Display Trace Log

In [ ]:
# Save structured results to JSON
results_file = "agentic_rag_results.json"
with open(results_file, "w") as f:
    json.dump(results, f, indent=2)
print(f"✅ Results saved to: {results_file}")

# Print the reasoning trace log
print(f"\n{'='*70}")
print("📋 REASONING TRACE LOG")
print('='*70)
with open(LOG_FILE, "r") as f:
    print(f.read())

## 13. Interactive Query Interface

Change the question below and re-run to test any query through the pipeline.

In [ ]:
custom_question = "What are the benefits of multi-agent orchestration in RAG systems?"

print(f"\n🔍 Running custom query: {custom_question}\n")
custom_result = run_agentic_rag(custom_question)
print(f"\n📍 Route  : {custom_result['route']}")
print(f"💬 Answer :\n{custom_result['answer']}")

## 14. Architecture Summary

| Component | Role | Technology |
|-----------|------|------------|
| Router Agent | Classifies query intent and selects retrieval path | CrewAI Agent + **Claude** |
| Retriever Agent | Executes retrieval and synthesises grounded answer | CrewAI Agent + **Claude** |
| Docling parser | Structure-aware PDF → Markdown | **Docling** (v2 API) |
| Local embeddings | Vectorise chunks with no API key | **HF all-MiniLM-L6-v2** |
| Vector store | Store + similarity search | **ChromaDB** (explicit) |
| Web search tool | Live web search for real-time info | Tavily Search API |
| Orchestration | Sequential task execution with context passing | CrewAI `Process.sequential` |
| Logging | Full reasoning trace to file + stdout | Python `logging` |

### Design Trade-offs

- **Local embeddings over a hosted API.** The original notebook died on an OpenAI `insufficient_quota` 429 during embedding. Local `all-MiniLM-L6-v2` has no key, no cost, and no remote endpoint — the most reliable choice for a live demo, at the price of a one-time model download and slightly lower embedding quality than a large hosted model.
- **Explicit Docling pipeline over `PDFSearchTool`.** More code, but every stage (parse → chunk → embed → store → search) is visible and swappable. For a PM audience this turns RAG from a magic box into five labeled steps.
- **Docling specifically.** Structure recognition (tables, reading order, multi-column layout) produces cleaner Markdown than a naive text dump, which chunks better and improves retrieval on a dense research paper.
- **Claude for both agents.** A single shared LLM keeps the demo simple; swap in a larger Claude for the Retriever if you want stronger synthesis than routing.
- **Sequential process.** The Router's label is passed as context to the Retriever, so the two must run in order.
